# Day 4: LangChain for Insurance Knowledge Assistants  
## Hands-on Lab Notebook

**Audience:** Beginner to Intermediate UiPath Developers, RPA Developers, Automation Engineers  
**Duration:** 4 Hours  
**Domain:** Insurance Policy FAQ, Customer Support, Claim Status, Policy Summary, and AI Helpdesk Automation

## Learning Outcomes

By the end of this notebook, participants will be able to:

- Understand FastAPI and Swagger UI step by step.
- Create a FastAPI endpoint for AI claim assistance.
- Understand LangChain and its insurance automation use cases.
- Use prompt templates and reusable chains.
- Use output parsers for structured responses.
- Build simple tools and agent-like routing.
- Build Policy FAQ, Customer Support, Claim Status and Policy Summary assistants.
- Complete the final Insurance Policy Assistant assignment.

# 0. How to Use This Notebook

Run the notebook step by step.

Recommended practice method:

1. Read the concept.
2. Run the code.
3. Modify the input.
4. Observe the output.
5. Complete each mini assignment.
6. Apply all components in the final Insurance Policy Assistant.

Environment variables are assumed to be configured:

```text
OPENROUTER_API_KEY=your_api_key_here
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
OPENROUTER_MODEL=openai/gpt-4.1-mini
```

# 1. Day 4 Training Flow

| Time | Topic |
|---|---|
| 30 min | FastAPI and Swagger UI |
| 30 min | AI Claim Assistance API |
| 30 min | LangChain Overview |
| 35 min | Prompt Templates and Chains |
| 35 min | Output Parsers |
| 35 min | Tools and Simple Agents |
| 40 min | Insurance Assistant Labs |
| 25 min | Final Assignment |

Trainer humour:  
**Prompt copy-paste is good for demos. LangChain is useful when the client says, “Can we scale this?”**

# 2. Install Required Libraries

Required packages:

- openai
- python-dotenv
- fastapi
- uvicorn
- pydantic
- langchain
- langchain-openai
- requests

In [ ]:
# Uncomment only if packages are missing.
!pip install openai python-dotenv fastapi uvicorn pydantic langchain langchain-openai requests

In [23]:
import fastapi
import uvicorn
import langchain
import langchain_openai
import requests

# 3. Environment and OpenAI-Compatible Client Setup

We will use an OpenAI-compatible client through OpenRouter.

This function will be reused in:

- FastAPI endpoint
- Insurance claim assistant
- Tool-based assistant
- LangChain labs

In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")

print("Base URL:", OPENROUTER_BASE_URL)
print("Model:", OPENROUTER_MODEL)
print("API Key Loaded:", bool(OPENROUTER_API_KEY))

client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY
)

def call_llm(prompt, system_message="You are an insurance automation assistant.", temperature=0.2):
    response = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

Base URL: https://openrouter.ai/api/v1
Model: openai/gpt-4.1-mini
API Key Loaded: True


# 4. FastAPI and Swagger UI — Step-by-Step

## What is FastAPI?

FastAPI is a Python framework used to create APIs.

For UiPath developers, it helps expose Python AI logic as an HTTP endpoint.

## What is Swagger UI?

FastAPI automatically creates interactive API documentation.

After running the API, open:

```text
http://127.0.0.1:8000/docs
```

Architecture:

```text
UiPath Robot → HTTP Request → FastAPI → LLM / LangChain → JSON Response → UiPath
```

Trainer humour:  
**Swagger UI is like Postman wearing a suit and tie.**

# 5. Lab: Create FastAPI Endpoint for AI Claim Assistance

Endpoint:

```text
POST /claim-assist
```

Input:

```json
{
  "claim_note": "raw insurance claim note"
}
```

Output:

```json
{
  "status": "success",
  "ai_response": "claim analysis"
}
```

In [3]:
fastapi_app_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

app = FastAPI(title="Insurance Helpdesk Bot Backend")

class ClaimRequest(BaseModel):
    claim_note: str

def call_llm(prompt):
    response = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {"role": "system", "content": "You are an insurance automation assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

@app.post("/claim-assist")
def claim_assist(request: ClaimRequest):
    prompt = f"""
    Analyze the insurance claim note and return:
    - Claim Summary
    - Claim Type
    - Risk Level
    - Missing Documents
    - Recommended Action

    Claim Note:
    {request.claim_note}
    """
    result = call_llm(prompt)
    return {"status": "success", "ai_response": result}
'''

Path("app.py").write_text(fastapi_app_code, encoding="utf-8")
print("app.py created.")
print("Run: uvicorn app:app --port 8001 --reload")
print("Open: http://127.0.0.1:8001/docs")

app.py created.
Run: uvicorn app:app --port 8001 --reload
Open: http://127.0.0.1:8001/docs


In [ ]:
!uvicorn app:app --port 8001 --reload

INFO:     Will watch for changes in these directories: ['/Users/surendra/Desktop/Allianz_Agentic_AI']
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)
INFO:     Started reloader process [93271] using WatchFiles
INFO:     Started server process [93273]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     127.0.0.1:56790 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:56791 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:56791 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:56882 - "POST /claim-assist HTTP/1.1" 200 OK
INFO:     127.0.0.1:57015 - "POST /claim-assist HTTP/1.1" 200 OK


## FastAPI Code Explanation

1. `FastAPI()` creates the API application.
2. `BaseModel` defines expected input JSON.
3. `ClaimRequest` expects `claim_note`.
4. `@app.post("/claim-assist")` creates the endpoint.
5. `call_llm()` sends the prompt to the LLM.
6. The endpoint returns JSON response.

## Hands-on Testing

Run in terminal:

```bash
uvicorn app:app --reload
```

Open Swagger UI:

```text
http://127.0.0.1:8000/docs
```

Use sample input:

```json
{
  "claim_note": "Customer submitted health claim for dengue treatment. Policy POL1001. Bill INR 85000. Hospital bill submitted."
}
```

In [ ]:
# Optional test after server is running.

import requests

sample_payload = {
    "claim_note": "Customer submitted health claim for dengue treatment. Policy POL1001. Bill INR 85000. Hospital bill submitted."
}

# response = requests.post("http://127.0.0.1:8000/claim-assist", json=sample_payload)
# print(response.status_code)
# print(response.json())

## Mini Assignment 1

Extend the FastAPI endpoint to accept:

```python
customer_type: str
```

Example values:

- Retail
- Corporate
- Senior Citizen

Modify the prompt so the recommended action considers customer type.

# 6. LangChain Overview

LangChain helps create structured LLM applications.

It provides:

- Prompt templates
- Chains
- Output parsers
- Tools
- Agents
- Memory
- Retrieval workflows

Insurance use cases:

- Policy FAQ assistant
- Policy summary assistant
- Claim status assistant
- Customer support assistant
- Structured claim assessment
- UiPath-ready JSON response generation

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field
from typing import List

llm = ChatOpenAI(
    model=OPENROUTER_MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0.2
)

print("LangChain model configured.")

LangChain model configured.


# 7. Prompt Templates

A prompt template is a reusable prompt with placeholders.

Example:

```text
Summarize this {policy_type} policy for {audience}.
```

Benefits:

- Reusability
- Consistency
- Governance
- Easier testing
- Suitable for automation

In [8]:
policy_text = """
Health Secure Plus covers hospitalization expenses up to INR 5,00,000.
Room rent is covered up to INR 5,000 per day.
Pre-hospitalization expenses are covered for 30 days.
Post-hospitalization expenses are covered for 60 days.
Cosmetic treatment and experimental procedures are excluded.
"""

policy_summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an insurance policy assistant. Explain clearly and do not invent policy details."),
    ("user", """
Summarize the following insurance policy for {audience}.

Policy Type: {policy_type}

Policy Text:
{policy_text}

Return:
- Coverage Summary
- Key Benefits
- Important Exclusions
- Customer Caution Note
""")
])

policy_summary_chain = policy_summary_prompt | llm | StrOutputParser()

# Uncomment after API key setup.
result = policy_summary_chain.invoke({
     "audience": "retail customer",
     "policy_type": "Health Insurance",
     "policy_text": policy_text
 })
print(result)

- Coverage Summary:
Health Secure Plus provides health insurance coverage with a maximum limit of INR 5,00,000 for hospitalization expenses. It includes room rent coverage up to INR 5,000 per day. Pre-hospitalization expenses are covered for 30 days before admission, and post-hospitalization expenses are covered for 60 days after discharge.

- Key Benefits:
1. Hospitalization expenses covered up to INR 5,00,000.
2. Room rent covered up to INR 5,000 per day.
3. Coverage for pre-hospitalization medical expenses for 30 days.
4. Coverage for post-hospitalization medical expenses for 60 days.

- Important Exclusions:
1. Cosmetic treatments are not covered.
2. Experimental medical procedures are excluded from coverage.

- Customer Caution Note:
Please review the policy terms carefully to understand the coverage limits and exclusions. Ensure that treatments related to cosmetic or experimental procedures are not expected to be reimbursed under this policy. For any specific medical conditions o

## Mini Assignment 2

Modify the policy summary template for:

1. Claims processor
2. Customer support executive
3. Sales team
4. Compliance reviewer

Observe how language and emphasis change.

# 8. Reusable Chains

A chain connects:

```text
Prompt Template → LLM → Output Parser
```

Insurance chains can be created for:

- Claim summary
- Policy summary
- Customer email
- Claim status
- Risk extraction

In [9]:
claim_summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an insurance claim operations assistant."),
    ("user", """
Create a claim summary from the note.

Return:
- Customer Name
- Policy Number
- Claim Type
- Claim Amount
- Current Status
- Next Action

Claim Note:
{claim_note}
""")
])

claim_summary_chain = claim_summary_prompt | llm | StrOutputParser()

health_claim_note = """
Customer Rajesh Sharma submitted a health insurance claim on 12 June 2026.
He was admitted to City Care Hospital for dengue fever treatment.
Total bill amount is INR 85,000. Policy number is POL1001.
Hospital bill, discharge summary and doctor prescription are submitted.
Claim is pending document verification.
"""

# Uncomment after API key setup.
print(claim_summary_chain.invoke({"claim_note": health_claim_note}))

- Customer Name: Rajesh Sharma  
- Policy Number: POL1001  
- Claim Type: Health Insurance  
- Claim Amount: INR 85,000  
- Current Status: Pending document verification  
- Next Action: Complete document verification process


## Lab 1: Create Reusable Policy Summary Chain

Input:

- policy_type
- policy_text
- audience

Expected output:

- Short summary
- Coverage limit
- Exclusions
- Action required

In [10]:
reusable_policy_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an insurance knowledge assistant."),
    ("user", """
Policy Type: {policy_type}
Audience: {audience}

Summarize the policy text.

Return:
- Short Summary
- Coverage Limit
- Exclusions
- Action Required

Policy Text:
{policy_text}
""")
])

reusable_policy_chain = reusable_policy_prompt | llm | StrOutputParser()

# Uncomment after API key setup.
print(reusable_policy_chain.invoke({
     "policy_type": "Health Insurance",
     "audience": "customer support team",
     "policy_text": policy_text
}))

- Short Summary:  
Health Secure Plus provides health insurance coverage for hospitalization expenses, including room rent, pre- and post-hospitalization costs, with specific exclusions.

- Coverage Limit:  
Hospitalization expenses up to INR 5,00,000; room rent up to INR 5,000 per day; pre-hospitalization expenses for 30 days; post-hospitalization expenses for 60 days.

- Exclusions:  
Cosmetic treatment and experimental procedures.

- Action Required:  
Ensure customers are informed about coverage limits and exclusions; verify claims against policy terms, especially for cosmetic and experimental treatments.


## Mini Assignment 3

Create a vehicle insurance policy summary chain.

Output should include:

- Own damage coverage
- Third-party coverage
- Exclusions
- Claim documents required

In [11]:
vehicle_policy_text = """
Comprehensive motor insurance covers own damage and third-party liability.
It covers accidental damage, theft and natural calamity damage.
Driving without a valid license, drunk driving and normal wear and tear are excluded.
Claim documents include policy copy, driving license, registration certificate, photos and repair estimate.
"""

vehicle_policy_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a vehicle insurance policy assistant."),
    ("user", """
Summarize this vehicle policy for {audience}.

Return:
- Own Damage Coverage
- Third-Party Coverage
- Exclusions
- Claim Documents Required

Policy Text:
{policy_text}
""")
])

vehicle_policy_chain = vehicle_policy_prompt | llm | StrOutputParser()

# Uncomment after API key setup.
print(vehicle_policy_chain.invoke({
     "audience": "claim processor",
     "policy_text": vehicle_policy_text
}))

- Own Damage Coverage: Covers accidental damage, theft, and natural calamity damage to the insured vehicle.
- Third-Party Coverage: Includes third-party liability protection.
- Exclusions: Driving without a valid license, drunk driving, and normal wear and tear.
- Claim Documents Required: Policy copy, driving license, vehicle registration certificate, photos of damage, and repair estimate.


# 9. Output Parsers and Structured Responses

UiPath prefers structured output.

Plain text is good for humans.  
JSON is better for automation.

LangChain output parsers help produce structured responses.

In [12]:
class ClaimAssessment(BaseModel):
    claim_type: str = Field(description="Type of insurance claim")
    risk_level: str = Field(description="Low, Medium or High")
    missing_documents: List[str] = Field(description="List of missing documents")
    recommended_action: str = Field(description="Next recommended action")


parser = JsonOutputParser(pydantic_object=ClaimAssessment)

claim_assessment_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an insurance claim assessment assistant. Return only valid JSON."),
    ("user", """
Assess the claim note.

{format_instructions}

Claim Note:
{claim_note}
""")
]).partial(format_instructions=parser.get_format_instructions())

claim_assessment_chain = claim_assessment_prompt | llm | parser

vehicle_claim_note = """
Customer Meena Joshi submitted a vehicle claim after a road accident.
Repair estimate is INR 2,40,000. Policy number is VEH2045.
Police report is not attached. Photos are unclear.
Policy started 5 days before accident.
"""

# Uncomment after API key setup.
result = claim_assessment_chain.invoke({"claim_note": vehicle_claim_note})
print(json.dumps(result, indent=4))

{
    "claim_type": "Vehicle",
    "risk_level": "High",
    "missing_documents": [
        "Police report",
        "Clear photos"
    ],
    "recommended_action": "Request police report and clear photos; verify policy start date and coverage eligibility before proceeding."
}


## Mini Assignment 4

Extend the structured output with:

- policy_number
- escalation_required
- routing_queue

Example queues:

- Health Claims Queue
- Vehicle Claims Queue
- Risk Review Queue
- Document Pending Queue

In [13]:
class ExtendedClaimAssessment(BaseModel):
    policy_number: str = Field(description="Policy number if available")
    claim_type: str = Field(description="Type of insurance claim")
    risk_level: str = Field(description="Low, Medium or High")
    missing_documents: List[str] = Field(description="List of missing documents")
    escalation_required: str = Field(description="Yes or No")
    routing_queue: str = Field(description="Target queue for automation")
    recommended_action: str = Field(description="Next recommended action")


extended_parser = JsonOutputParser(pydantic_object=ExtendedClaimAssessment)

extended_claim_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an insurance claim routing assistant. Return only valid JSON."),
    ("user", """
Analyze the claim note and create routing output.

{format_instructions}

Claim Note:
{claim_note}
""")
]).partial(format_instructions=extended_parser.get_format_instructions())

extended_claim_chain = extended_claim_prompt | llm | extended_parser

# Uncomment after API key setup.
print(extended_claim_chain.invoke({"claim_note": vehicle_claim_note}))

{'policy_number': 'VEH2045', 'claim_type': 'Vehicle', 'risk_level': 'High', 'missing_documents': ['Police report', 'Clear photos'], 'escalation_required': 'Yes', 'routing_queue': 'Vehicle Claims - High Risk', 'recommended_action': 'Request missing documents and escalate for fraud review due to recent policy start date'}


# 10. Tools and Simple Agent Examples

A tool is a function that an AI assistant can use.

Insurance tools:

- Policy lookup
- Claim status lookup
- Premium status lookup
- Document checklist
- Customer profile lookup

We will build beginner-friendly tool routing.

In [14]:
policy_database = {
    "POL1001": {
        "customer_name": "Rajesh Sharma",
        "policy_type": "Health",
        "coverage_amount": 500000,
        "policy_status": "Active",
        "premium_status": "Paid"
    },
    "VEH2045": {
        "customer_name": "Meena Joshi",
        "policy_type": "Vehicle",
        "coverage_amount": 300000,
        "policy_status": "Active",
        "premium_status": "Pending"
    }
}

claim_status_database = {
    "CLM9001": {
        "policy_number": "POL1001",
        "claim_status": "Pending Document Verification",
        "last_update": "Hospital bill received. Discharge summary under review."
    },
    "CLM9002": {
        "policy_number": "VEH2045",
        "claim_status": "Pending Documents",
        "last_update": "Police report and clear photos are required."
    }
}

def policy_lookup_tool(policy_number):
    return policy_database.get(policy_number, {"error": "Policy not found"})

def claim_status_tool(claim_id):
    return claim_status_database.get(claim_id, {"error": "Claim not found"})

def premium_status_tool(policy_number):
    policy = policy_database.get(policy_number)
    if not policy:
        return {"error": "Policy not found"}
    premium_status = policy["premium_status"]
    action = "Policy is eligible for normal processing." if premium_status == "Paid" else "Premium is pending. Ask customer to complete payment."
    return {
        "policy_number": policy_number,
        "premium_status": premium_status,
        "recommended_action": action
    }

print(policy_lookup_tool("POL1001"))
print(claim_status_tool("CLM9002"))
print(premium_status_tool("VEH2045"))

{'customer_name': 'Rajesh Sharma', 'policy_type': 'Health', 'coverage_amount': 500000, 'policy_status': 'Active', 'premium_status': 'Paid'}
{'policy_number': 'VEH2045', 'claim_status': 'Pending Documents', 'last_update': 'Police report and clear photos are required.'}
{'policy_number': 'VEH2045', 'premium_status': 'Pending', 'recommended_action': 'Premium is pending. Ask customer to complete payment.'}


In [15]:
def simple_insurance_tool_router(user_question):
    question = user_question.lower()

    if "premium" in question and "pol1001" in question:
        return premium_status_tool("POL1001")
    if "premium" in question and "veh2045" in question:
        return premium_status_tool("VEH2045")
    if "policy" in question and "pol1001" in question:
        return policy_lookup_tool("POL1001")
    if "policy" in question and "veh2045" in question:
        return policy_lookup_tool("VEH2045")
    if "claim" in question and "clm9001" in question:
        return claim_status_tool("CLM9001")
    if "claim" in question and "clm9002" in question:
        return claim_status_tool("CLM9002")

    return {"message": "No matching tool found. Route to human support."}

print(simple_insurance_tool_router("What is the status of claim CLM9002?"))

{'policy_number': 'VEH2045', 'claim_status': 'Pending Documents', 'last_update': 'Police report and clear photos are required.'}


## Lab 2: Tool-Based Customer Answer

Use a tool result and LLM to generate a customer-friendly answer.

Example:

```text
User asks: What is the status of claim CLM9002?
Tool returns: Pending Documents
LLM generates: Professional customer response
```

In [16]:
def answer_using_tool_result(user_question):
    tool_result = simple_insurance_tool_router(user_question)

    prompt = f"""
You are an insurance customer support assistant.

Create a clear customer-friendly answer based on the tool result.

Rules:
- Do not invent details.
- If data is missing, ask customer to contact support.
- Keep the tone professional.

User Question:
{user_question}

Tool Result:
{json.dumps(tool_result, indent=2)}
"""
    return call_llm(prompt)

# Uncomment after API key setup.
print(answer_using_tool_result("What is the status of claim CLM9002?"))

The status of claim CLM9002 is "Pending Documents." We are currently waiting to receive the police report and clear photos to proceed with your claim. If you need further assistance, please contact our support team.


## Mini Assignment 5

Add one more tool:

```python
document_checklist_tool(claim_type)
```

It should return required documents for:

- Health claim
- Vehicle claim
- Travel claim
- Property claim

# 11. Lab 3: Policy FAQ Assistant

The assistant should answer only from the given policy text.

Questions:

- Is room rent covered?
- Are cosmetic treatments covered?
- Are pre-hospitalization expenses covered?
- Is dental treatment covered?

In [17]:
policy_faq_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an insurance policy FAQ assistant. Answer only from provided policy text."),
    ("user", """
Policy Text:
{policy_text}

Customer Question:
{question}

Rules:
- Answer only from the policy text.
- If not available, say information is not available.
- Keep the answer simple and professional.
""")
])

policy_faq_chain = policy_faq_prompt | llm | StrOutputParser()

faq_questions = [
    "Is room rent covered?",
    "Are cosmetic treatments covered?",
    "Are pre-hospitalization expenses covered?",
    "Is dental treatment covered?"
]

# Uncomment after API key setup.
for q in faq_questions:
     print("Question:", q)
     print(policy_faq_chain.invoke({"policy_text": policy_text, "question": q}))
     print("-" * 60)

Question: Is room rent covered?
Yes, room rent is covered up to INR 5,000 per day.
------------------------------------------------------------
Question: Are cosmetic treatments covered?
Cosmetic treatments are excluded from coverage under Health Secure Plus.
------------------------------------------------------------
Question: Are pre-hospitalization expenses covered?
Yes, pre-hospitalization expenses are covered for 30 days under the Health Secure Plus policy.
------------------------------------------------------------
Question: Is dental treatment covered?
Information is not available regarding coverage for dental treatment in the policy text provided.
------------------------------------------------------------


# 12. Lab 4: Customer Support Assistant

Rules:

- Do not promise claim approval.
- Mention current known status.
- Ask for missing documents if required.
- Keep it concise and professional.

In [18]:
customer_support_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional insurance customer support assistant."),
    ("user", """
Draft a polite customer email reply.

Rules:
- Do not promise claim approval.
- Mention current known status.
- Ask for missing documents if required.
- Keep it concise and professional.

Customer Message:
{customer_message}

Claim Context:
{claim_context}
""")
])

customer_support_chain = customer_support_prompt | llm | StrOutputParser()

customer_message = """
Dear Team,
I submitted my hospital claim last week but have not received any update.
Please let me know whether my claim is approved.
Regards,
Rajesh Sharma
"""

# Uncomment after API key setup.
print(customer_support_chain.invoke({
     "customer_message": customer_message,
     "claim_context": health_claim_note
 }))

Subject: Update on Your Hospital Claim Submission

Dear Mr. Sharma,

Thank you for reaching out. Your claim submitted on 12 June 2026 for treatment at City Care Hospital is currently under document verification. We have received the hospital bill, discharge summary, and doctor’s prescription.

If any additional information or documents are required, we will contact you promptly. We appreciate your patience during this process.

Best regards,  
[Your Name]  
Customer Support Team  
[Insurance Company Name]


## Mini Assignment 6

Create an SMS chain.

Rules:

- Maximum 30 words
- No approval promise
- Clear next action

In [19]:
sms_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an insurance support SMS assistant."),
    ("user", """
Create an SMS update under 30 words.

Rules:
- Do not promise claim approval.
- Mention next action.

Customer Message:
{customer_message}

Claim Context:
{claim_context}
""")
])

sms_chain = sms_prompt | llm | StrOutputParser()

# Uncomment after API key setup.
print(sms_chain.invoke({
     "customer_message": customer_message,
     "claim_context": health_claim_note
 }))

Dear Rajesh, your claim is under document verification. We will update you once the review is complete. Thank you for your patience.


# 13. Lab 5: Claim Status Assistant

This combines:

1. Claim ID
2. Claim status lookup tool
3. LLM-generated explanation

In [ ]:
def claim_status_assistant(claim_id):
    status_data = claim_status_tool(claim_id)
    prompt = f"""
You are an insurance claim status assistant.

Create a customer-friendly status update.

Rules:
- Use only provided status data.
- Do not promise approval.
- Mention required next action.

Claim ID:
{claim_id}

Status Data:
{json.dumps(status_data, indent=2)}
"""
    return call_llm(prompt)

# Uncomment after API key setup.
# print(claim_status_assistant("CLM9002"))

## Mini Assignment 7

Create an internal claim processor version.

Output:

- Current status
- Pending action
- Suggested queue
- Customer communication note

In [ ]:
def internal_claim_status_assistant(claim_id):
    status_data = claim_status_tool(claim_id)
    prompt = f"""
You are an insurance claim operations assistant.

Create an internal claim processor note.

Return:
- Current Status
- Pending Action
- Suggested Queue
- Customer Communication Note

Claim ID:
{claim_id}

Status Data:
{json.dumps(status_data, indent=2)}
"""
    return call_llm(prompt)

# Uncomment after API key setup.
# print(internal_claim_status_assistant("CLM9002"))

# 14. Extra Practice Example 1: Policy Comparison Assistant

Use case: Compare two health policies for customer support.

In [ ]:
basic_health_policy = """
Basic Health Plan covers hospitalization up to INR 2,00,000.
Room rent is covered up to INR 2,000 per day.
Pre-hospitalization is covered for 15 days.
Post-hospitalization is covered for 30 days.
"""

premium_health_policy = """
Premium Health Plan covers hospitalization up to INR 10,00,000.
Room rent is covered up to INR 8,000 per day.
Pre-hospitalization is covered for 45 days.
Post-hospitalization is covered for 90 days.
Annual health checkup is included.
"""

policy_comparison_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an insurance policy comparison assistant."),
    ("user", """
Compare the two health policies.

Return:
- Coverage Difference
- Room Rent Difference
- Pre/Post Hospitalization Difference
- Best Suitable Customer Type
- Caution Note

Policy A:
{policy_a}

Policy B:
{policy_b}
""")
])

policy_comparison_chain = policy_comparison_prompt | llm | StrOutputParser()

# Uncomment after API key setup.
# print(policy_comparison_chain.invoke({
#     "policy_a": basic_health_policy,
#     "policy_b": premium_health_policy
# }))

## Assignment Example 1

Create comparison chain for:

- Standard motor policy
- Comprehensive motor policy

Output:

- Coverage difference
- Exclusions
- Documents required
- Recommendation note

# 15. Extra Practice Example 2: Underwriting Support Chain

AI should assist underwriters but not approve or reject applicants.

In [ ]:
underwriting_note = """
Applicant is 52 years old and applying for a health policy of INR 10,00,000.
Medical history includes hypertension and diabetes for the last 8 years.
Applicant is a non-smoker. BMI is 31.
Previous hospitalization occurred two years ago.
"""

underwriting_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an insurance underwriting support assistant. Do not approve or reject applicants."),
    ("user", """
Create structured underwriting support notes.

Return:
- Applicant Summary
- Medical Risk Factors
- Missing Information
- Recommended Underwriter Review Points

Applicant Details:
{applicant_details}
""")
])

underwriting_chain = underwriting_prompt | llm | StrOutputParser()

# Uncomment after API key setup.
# print(underwriting_chain.invoke({"applicant_details": underwriting_note}))

## Assignment Example 2

Create structured parser version for underwriting.

Fields:

- applicant_age
- medical_risk_factors
- lifestyle_factors
- missing_information
- recommended_review

# 16. Extra Practice Example 3: Complaint Triage Assistant

Use case: Prioritize insurance support complaints.

In [ ]:
complaint_text = """
I have called your support team five times and nobody has given me a clear answer.
My father's hospital claim is pending for three weeks. We urgently need reimbursement.
Please escalate this immediately.
"""

complaint_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an insurance support triage assistant."),
    ("user", """
Classify this complaint.

Return:
- Sentiment
- Urgency
- Category
- Escalation Required
- Suggested Response Tone
- Reason

Complaint:
{complaint}
""")
])

complaint_chain = complaint_prompt | llm | StrOutputParser()

# Uncomment after API key setup.
# print(complaint_chain.invoke({"complaint": complaint_text}))

## Assignment Example 3

Create JSON output for complaint triage.

Fields:

- sentiment
- urgency
- category
- escalation_required
- target_queue
- suggested_response_tone

# 17. Extra Practice Example 4: Document Checklist Assistant

Use case: Claims get delayed because documents are missing.

In [ ]:
document_checklist_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an insurance document verification assistant."),
    ("user", """
Identify submitted and missing documents.

Return:
- Claim Type
- Submitted Documents
- Missing Documents
- Why Missing Documents Are Needed
- Next Action

Claim Note:
{claim_note}
""")
])

document_checklist_chain = document_checklist_prompt | llm | StrOutputParser()

# Uncomment after API key setup.
# print(document_checklist_chain.invoke({"claim_note": vehicle_claim_note}))

## Assignment Example 4

Create reusable document checklist chains for:

- Health claim
- Vehicle claim
- Travel claim
- Property claim

# 18. Final Assignment: Insurance Policy Assistant using LangChain

## Objective

Build an Insurance Policy Assistant that can:

1. Answer policy FAQ questions.
2. Summarize policy text.
3. Generate customer-friendly explanations.
4. Create structured claim assessment output.
5. Check claim status using a tool.
6. Generate support responses.
7. Return JSON where automation needs structured output.

In [ ]:
class InsurancePolicyAssistant:
    def __init__(self, llm):
        self.llm = llm
        self.policy_faq_chain = policy_faq_chain
        self.policy_summary_chain = reusable_policy_chain
        self.customer_support_chain = customer_support_chain
        self.claim_assessment_chain = claim_assessment_chain

    def answer_policy_question(self, policy_text, question):
        return self.policy_faq_chain.invoke({
            "policy_text": policy_text,
            "question": question
        })

    def summarize_policy(self, policy_type, audience, policy_text):
        return self.policy_summary_chain.invoke({
            "policy_type": policy_type,
            "audience": audience,
            "policy_text": policy_text
        })

    def generate_customer_reply(self, customer_message, claim_context):
        return self.customer_support_chain.invoke({
            "customer_message": customer_message,
            "claim_context": claim_context
        })

    def assess_claim(self, claim_note):
        return self.claim_assessment_chain.invoke({
            "claim_note": claim_note
        })

    def get_claim_status(self, claim_id):
        return claim_status_assistant(claim_id)


assistant = InsurancePolicyAssistant(llm)
print("InsurancePolicyAssistant class created.")

In [ ]:
# Final Assignment Testing
# Uncomment after API key setup.

# print(assistant.answer_policy_question(policy_text, "Is room rent covered?"))

# print(assistant.summarize_policy(
#     policy_type="Health Insurance",
#     audience="customer support executive",
#     policy_text=policy_text
# ))

# print(assistant.generate_customer_reply(
#     customer_message=customer_message,
#     claim_context=health_claim_note
# ))

# print(assistant.assess_claim(vehicle_claim_note))

# print(assistant.get_claim_status("CLM9002"))

# 19. Optional Advanced Assignment: FastAPI + LangChain Assistant

Create endpoint:

```text
POST /policy-assistant
```

Input:

```json
{
  "task": "policy_faq",
  "question": "Is room rent covered?",
  "policy_text": "..."
}
```

Possible tasks:

- policy_faq
- policy_summary
- customer_reply
- claim_assessment
- claim_status

This makes the assistant callable from UiPath.

# 20. Review and Discussion Questions

1. Why are prompt templates better than copy-paste prompts?
2. What is the difference between a chain and an agent?
3. Why do UiPath workflows prefer structured JSON?
4. What can go wrong if a policy assistant invents answers?
5. Where should human review be mandatory?
6. Which insurance workflows are good candidates for LangChain?
7. How can FastAPI expose LangChain assistants to UiPath?
8. What should be logged for audit and compliance?

# 21. Day 4 Summary

You completed:

- FastAPI and Swagger UI walkthrough
- AI claim assistance endpoint
- LangChain overview
- Prompt templates
- Reusable chains
- Output parsers
- Structured JSON responses
- Simple tools and agent-like routing
- Policy FAQ assistant
- Customer support assistant
- Claim status assistant
- Policy summary chain
- Extra practice examples
- Final Insurance Policy Assistant assignment

## Next Step

This Day 4 foundation supports:

- RAG-based policy assistants
- LangGraph multi-agent insurance workflows
- MCP tool integrations
- UiPath + AI backend integration
- Enterprise insurance automation capstone